In [30]:
import pandas as pd

# Paths to the files inside the Kaggle dataset folder
train_path = "/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/train_data.txt"
test_path = "/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/test_data.txt"
test_solution_path = "/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/test_data_solution.txt"

train_path, test_path, test_solution_path


('/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/train_data.txt',
 '/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/test_data.txt',
 '/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/test_data_solution.txt')

In [31]:
# Read all lines from train_data.txt
with open(train_path, "r", encoding="utf-8") as f:
    train_lines = f.readlines()

# Show first 5 lines to check format
train_lines[:5]


['1 ::: Oscar et la dame rose (2009) ::: drama ::: Listening in to a conversation between his doctor and parents, 10-year-old Oscar learns what nobody has the courage to tell him. He only has a few weeks to live. Furious, he refuses to speak to anyone except straight-talking Rose, the lady in pink he meets on the hospital stairs. As Christmas approaches, Rose uses her fantastical experiences as a professional wrestler, her imagination, wit and charm to allow Oscar to live life and love to the full, in the company of his friends Pop Corn, Einstein, Bacon and childhood sweetheart Peggy Blue.\n',
 '2 ::: Cupid (1997) ::: thriller ::: A brother and sister with a past incestuous relationship have a current murderous relationship. He murders the women who reject him and she murders the women who get too close to him.\n',
 '3 ::: Young, Wild and Wonderful (1980) ::: adult ::: As the bus empties the students for their field trip to the Museum of Natural History, little does the tour guide susp

In [32]:
train_rows = []

for line in train_lines:
    line = line.strip()
    if not line:
        continue
    
    parts = line.split(" ::: ")
    
    # train file has 4 parts: ID, TITLE, GENRE, DESCRIPTION
    if len(parts) == 4:
        id_, title, genre, description = parts
        train_rows.append((id_, title, genre, description))

train_df = pd.DataFrame(train_rows, columns=["id", "title", "genre", "description"])

train_df.head(), len(train_df)


(  id                             title     genre  \
 0  1      Oscar et la dame rose (2009)     drama   
 1  2                      Cupid (1997)  thriller   
 2  3  Young, Wild and Wonderful (1980)     adult   
 3  4             The Secret Sin (1915)     drama   
 4  5            The Unrecovered (2007)     drama   
 
                                          description  
 0  Listening in to a conversation between his doc...  
 1  A brother and sister with a past incestuous re...  
 2  As the bus empties the students for their fiel...  
 3  To help their unemployed father make ends meet...  
 4  The film's title refers not only to the un-rec...  ,
 54214)

In [33]:
# Read all lines from test_data.txt
with open(test_path, "r", encoding="utf-8") as f:
    test_lines = f.readlines()

test_rows = []

for line in test_lines:
    line = line.strip()
    if not line:
        continue
    
    parts = line.split(" ::: ")
    
    # test file has 3 parts: ID, TITLE, DESCRIPTION
    if len(parts) == 3:
        id_, title, description = parts
        test_rows.append((id_, title, description))

test_df = pd.DataFrame(test_rows, columns=["id", "title", "description"])

test_df.head(), len(test_df)


(  id                        title  \
 0  1         Edgar's Lunch (1998)   
 1  2     La guerra de papá (1977)   
 2  3  Off the Beaten Track (2010)   
 3  4       Meu Amigo Hindu (2015)   
 4  5            Er nu zhai (1955)   
 
                                          description  
 0  L.R. Brane loves his life - his car, his apart...  
 1  Spain, March 1964: Quico is a very naughty chi...  
 2  One year in the life of Albin and his family o...  
 3  His father has died, he hasn't spoken with his...  
 4  Before he was known internationally as a marti...  ,
 54200)

In [34]:
# Read all lines from test_data_solution.txt
with open(test_solution_path, "r", encoding="utf-8") as f:
    solution_lines = f.readlines()

solution_rows = []

for line in solution_lines:
    line = line.strip()
    if not line:
        continue

    # Split using ':::' and strip spaces around each part
    parts = [p.strip() for p in line.split(":::")]

    # solution file should have: ID, TITLE, GENRE
    if len(parts) >= 3:
        id_, title, genre = parts[0], parts[1], parts[2]
        solution_rows.append((id_, title, genre))

solution_df = pd.DataFrame(solution_rows, columns=["id", "title", "genre"])

solution_df.head(), len(solution_df)


(  id                        title        genre
 0  1         Edgar's Lunch (1998)     thriller
 1  2     La guerra de papá (1977)       comedy
 2  3  Off the Beaten Track (2010)  documentary
 3  4       Meu Amigo Hindu (2015)        drama
 4  5            Er nu zhai (1955)        drama,
 54200)

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [36]:
# Features (movie descriptions) and labels (genres)
X = train_df["description"]
y = train_df["genre"]

# Train/validation split (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

len(X_train), len(X_val)


(43371, 10843)

In [37]:
# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(stop_words="english", max_features=20000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

# Create and train Logistic Regression classifier
model = LogisticRegression(max_iter=500, class_weight="balanced")

model.fit(X_train_tfidf, y_train)

# Predict on validation set
y_val_pred = model.predict(X_val_tfidf)

# Check accuracy and detailed report
val_accuracy = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy:", val_accuracy)

print("\nClassification Report:\n")
print(classification_report(y_val, y_val_pred))


Validation Accuracy: 0.4838144424974638

Classification Report:

              precision    recall  f1-score   support

      action       0.32      0.48      0.39       263
       adult       0.40      0.74      0.52       118
   adventure       0.22      0.38      0.28       155
   animation       0.18      0.25      0.21       100
   biography       0.07      0.15      0.09        53
      comedy       0.60      0.47      0.53      1490
       crime       0.14      0.33      0.20       101
 documentary       0.78      0.59      0.67      2619
       drama       0.70      0.39      0.50      2723
      family       0.15      0.32      0.21       157
     fantasy       0.13      0.23      0.17        65
   game-show       0.73      0.69      0.71        39
     history       0.11      0.29      0.15        49
      horror       0.58      0.70      0.63       441
       music       0.39      0.79      0.52       146
     musical       0.10      0.18      0.13        55
     mystery    

In [38]:
# TF-IDF for test descriptions
X_test = test_df["description"]
X_test_tfidf = vectorizer.transform(X_test)

# Predict genres for test data
test_pred = model.predict(X_test_tfidf)

# Add predictions to test_df
test_df["predicted_genre"] = test_pred

test_df.head()


,id,title,description,predicted_genre
0,1,Edgar's Lunch (1998),"L.R. Brane loves his life - his car, his apart...",short
1,2,La guerra de papá (1977),"Spain, March 1964: Quico is a very naughty chi...",drama
2,3,Off the Beaten Track (2010),One year in the life of Albin and his family o...,documentary
3,4,Meu Amigo Hindu (2015),"His father has died, he hasn't spoken with his...",drama
4,5,Er nu zhai (1955),Before he was known internationally as a marti...,drama


In [39]:
# CLEAN ID columns: string, remove BOM, strip spaces

test_df["id"] = (
    test_df["id"]
    .astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

solution_df["id"] = (
    solution_df["id"]
    .astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

print("First 5 ids in test_df:", test_df["id"].head().tolist())
print("First 5 ids in solution_df:", solution_df["id"].head().tolist())

# Merge predictions with true genres by CLEANED id
merged = test_df.merge(
    solution_df[["id", "genre"]].rename(columns={"genre": "true_genre"}),
    on="id",
    how="inner"
)

print("len(test_df)     =", len(test_df))
print("len(solution_df) =", len(solution_df))
print("len(merged)      =", len(merged))

merged.head()


First 5 ids in test_df: ['1', '2', '3', '4', '5']
First 5 ids in solution_df: ['1', '2', '3', '4', '5']
len(test_df)     = 54200
len(solution_df) = 54200
len(merged)      = 54200


,id,title,description,predicted_genre,true_genre
0,1,Edgar's Lunch (1998),"L.R. Brane loves his life - his car, his apart...",short,thriller
1,2,La guerra de papá (1977),"Spain, March 1964: Quico is a very naughty chi...",drama,comedy
2,3,Off the Beaten Track (2010),One year in the life of Albin and his family o...,documentary,documentary
3,4,Meu Amigo Hindu (2015),"His father has died, he hasn't spoken with his...",drama,drama
4,5,Er nu zhai (1955),Before he was known internationally as a marti...,drama,drama


In [40]:
from sklearn.metrics import accuracy_score

if len(merged) > 0:
    test_accuracy = accuracy_score(
        merged["true_genre"],
        merged["predicted_genre"]
    )
    print("Test Set Accuracy (by id):", test_accuracy)
else:
    print("Test Set Accuracy cannot be computed because merged is empty.")


Test Set Accuracy (by id): 0.48682656826568266
